# Эксперименты 1–2: слово или псевдослово

Описание методики и результатов — в `README.md` репозитория.

## 1. Подготовить среду

In [ ]:
%pip -q install "transformers==4.48.3" "accelerate>=1.2.0" "bitsandbytes>=0.45.0" "wordfreq>=3.1.1" pymorphy3 pymorphy3-dicts-ru scikit-learn scipy pandas matplotlib tqdm

In [ ]:
import torch
import transformers
import huggingface_hub
import bitsandbytes
import pymorphy3

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("pymorphy3:", pymorphy3.__version__)

## 2. Импорты и настройки эксперимента

In [ ]:
import os
import re
import json
import random
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.auto import tqdm
from wordfreq import top_n_list, zipf_frequency
import pymorphy3
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SEED = 42
N_SEEDS = 5
SEEDS = [SEED + i for i in range(N_SEEDS)]
MODEL_NAME = "IlyaGusev/saiga_llama3_8b"
TOP_N_WORDS = 100_000
N_WORDS = 1_200
MIN_TOKENS = 2
MAX_TOKENS = 4
TOKENIZE_BATCH_SIZE = 2_048
MODEL_BATCH_SIZE = 8
K_NEIGHBORS = 4
TEST_SIZE = 0.20
MAX_ATTEMPTS = 20_000
N_PERMUTATIONS = 50
RUN_LINEAR_PROBE = True
LINEAR_PCA_DIM = 256
USE_4BIT = True
RESULT_DIR = "/content/saiga_russian_word_nonword_results_v2"
os.makedirs(RESULT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Нужен GPU: в Colab выберите Runtime → Change runtime type → GPU.")
print("Сиды для псевдослов и разбиений:", SEEDS)

## 3. Загрузить токенизатор и модель

In [ ]:
assert torch.cuda.is_available()

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
PAD_ID = tokenizer.pad_token_id
BOS_ID = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id
if BOS_ID is None or PAD_ID is None:
    raise ValueError("У токенизатора не нашлись BOS/EOS/PAD token IDs.")

load_kwargs = {
    "device_map": "auto",
    "torch_dtype": COMPUTE_DTYPE,
    "low_cpu_mem_usage": True,
}
if USE_4BIT:
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()

print("Loaded:", MODEL_NAME)
print("Transformer layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("BOS ID:", BOS_ID, "PAD ID:", PAD_ID)

## 4. Собрать и почистить список настоящих русских слов

In [ ]:
morph = pymorphy3.MorphAnalyzer()
PROPER_OR_ABBR = {"Name", "Surn", "Patr", "Geox", "Orgn", "Trad", "Abbr", "Init"}


def is_common_dictionary_word(word):
    if not morph.word_is_known(word):
        return False
    best = morph.parse(word)[0]
    return not any(g in best.tag for g in PROPER_OR_ABBR)


raw_words = top_n_list("ru", TOP_N_WORDS)

candidate_words = []
seen_words = set()
for raw_word in raw_words:
    word = unicodedata.normalize("NFC", raw_word.strip().lower())
    if not re.fullmatch(r"[а-яё]+", word):
        continue
    if len(word) < 3 or word in seen_words:
        continue
    seen_words.add(word)
    candidate_words.append(word)

all_frequency_forms = set(candidate_words)

filtered_words = []
removed_examples = []
for word in tqdm(candidate_words, desc="Dictionary filter (pymorphy3)"):
    if is_common_dictionary_word(word):
        filtered_words.append(word)
    elif len(removed_examples) < 25:
        removed_examples.append(word)

print("Буквенных слов из wordfreq:", len(candidate_words))
print("После словарной чистки:", len(filtered_words))
print("Примеры удалённых:", ", ".join(removed_examples))

eligible = []
seen_token_sequences = set()
for start in tqdm(range(0, len(filtered_words), TOKENIZE_BATCH_SIZE), desc="Tokenizing Russian words"):
    batch_words = filtered_words[start:start + TOKENIZE_BATCH_SIZE]
    encoded = tokenizer([" " + w for w in batch_words], add_special_tokens=False, padding=False)
    for word, ids in zip(batch_words, encoded["input_ids"]):
        if not (MIN_TOKENS <= len(ids) <= MAX_TOKENS):
            continue
        seq = tuple(int(x) for x in ids)
        if seq in seen_token_sequences:
            continue
        seen_token_sequences.add(seq)
        eligible.append({"word": word, "token_ids": list(seq), "num_tokens": len(seq)})

print("Кандидатов, разбитых на 2–4 токена:", len(eligible))
print("Распределение по числу токенов:", dict(sorted(Counter(x["num_tokens"] for x in eligible).items())))

if len(eligible) < N_WORDS:
    raise ValueError(
        f"Нашлось только {len(eligible)} подходящих слов при N_WORDS={N_WORDS}. "
        "Увеличьте TOP_N_WORDS или уменьшите N_WORDS."
    )

rng_words = np.random.default_rng(SEED)
eligible_shuffled = list(eligible)
rng_words.shuffle(eligible_shuffled)
selected_words = eligible_shuffled[:N_WORDS]
print("Выбрано настоящих слов:", len(selected_words))
print("Их распределение по числу токенов:", dict(sorted(Counter(x["num_tokens"] for x in selected_words).items())))

## 5. Создать псевдослова

In [ ]:
def is_canonical(ids):
    text = tokenizer.decode(list(ids), clean_up_tokenization_spaces=False)
    return tokenizer.encode(text, add_special_tokens=False) == list(ids)


def decoded_word(ids):
    return tokenizer.decode(list(ids), clean_up_tokenization_spaces=False).strip()


selected_by_length = defaultdict(list)
for item in selected_words:
    selected_by_length[item["num_tokens"]].append(item["token_ids"])

known_real_sequences = {tuple(item["token_ids"]) for item in eligible}


def make_pseudowords(set_name, seed, apply_filters):
    rng = np.random.default_rng(seed)
    seen = set()
    out = []
    attempt_stats = {}
    for length in sorted(selected_by_length):
        group = selected_by_length[length]
        pools = [np.array([seq[pos] for seq in group], dtype=np.int64) for pos in range(length)]
        tried = rejected_canon = rejected_text = rejected_dict = 0
        made = 0
        for _ in tqdm(range(len(group)), desc=f"{set_name}, length {length}", leave=False):
            for attempt in range(MAX_ATTEMPTS):
                candidate = tuple(int(rng.choice(pool)) for pool in pools)
                tried += 1
                if candidate in known_real_sequences or candidate in seen:
                    continue
                text = decoded_word(candidate)
                if apply_filters:
                    if not re.fullmatch(r"[а-яё]+", text):
                        rejected_text += 1
                        continue
                    if not is_canonical(candidate):
                        rejected_canon += 1
                        continue
                    if text in all_frequency_forms or morph.word_is_known(text):
                        rejected_dict += 1
                        continue
                seen.add(candidate)
                out.append({
                    "text": text,
                    "token_ids": list(candidate),
                    "num_tokens": length,
                    "label": 0,
                    "kind": "pseudoword",
                    "set_name": set_name,
                    "seed": seed,
                })
                made += 1
                break
        attempt_stats[length] = {
            "нужно": len(group), "создано": made, "попыток": tried,
            "отказ: не буквы": rejected_text, "отказ: неканон.": rejected_canon,
            "отказ: словарь": rejected_dict,
        }
    return out, attempt_stats


real_records = [{
    "text": item["word"],
    "token_ids": item["token_ids"],
    "num_tokens": item["num_tokens"],
    "label": 1,
    "kind": "word",
    "set_name": "real",
    "seed": SEED,
} for item in selected_words]

PSEUDO_SETS = [(f"canon_s{s}", s, True) for s in SEEDS] + [(f"nofilter_s{SEEDS[0]}", SEEDS[0], False)]
CANON_SETS = [name for name, _, filt in PSEUDO_SETS if filt]
NOFILTER_SET = f"nofilter_s{SEEDS[0]}"

pseudo_records = []
generation_rows = []
for set_name, seed, apply_filters in PSEUDO_SETS:
    recs, st = make_pseudowords(set_name, seed, apply_filters)
    pseudo_records.extend(recs)
    for length, row in st.items():
        generation_rows.append({"set_name": set_name, "num_tokens": length, **row})

generation_df = pd.DataFrame(generation_rows)
print("Статистика генерации псевдослов:")
display(generation_df)

nofilter_recs = [r for r in pseudo_records if r["set_name"] == NOFILTER_SET]
share_noncanon = np.mean([not is_canonical(r["token_ids"]) for r in nofilter_recs])
share_bad_text = np.mean([not re.fullmatch(r"[а-яё]+", r["text"]) for r in nofilter_recs])
print(f"В наборе {NOFILTER_SET} (метод первой версии):")
print(f"  неканоническая токенизация: {share_noncanon:.1%}")
print(f"  строка не из русских букв (разрезанные байты): {share_bad_text:.1%}")

records = real_records + pseudo_records
examples_df = pd.DataFrame([{**{k: v for k, v in r.items() if k != "token_ids"},
                             "token_ids": json.dumps(r["token_ids"])} for r in records])
examples_path = os.path.join(RESULT_DIR, "russian_words_and_pseudowords.csv")
examples_df.to_csv(examples_path, index=False)
generation_df.to_csv(os.path.join(RESULT_DIR, "pseudoword_generation_stats.csv"), index=False)
print("Список примеров сохранён:", examples_path)
display(examples_df.groupby(["set_name", "num_tokens"]).size().unstack(fill_value=0))
display(examples_df[examples_df["set_name"].isin(["real", CANON_SETS[0], NOFILTER_SET])]
        .groupby("set_name").sample(6, random_state=SEED)[["set_name", "text", "num_tokens", "token_ids"]])

## 6. Извлечь hidden states на каждом слое

In [ ]:
DEVICE = torch.device("cuda:0")
N_STATES = model.config.num_hidden_layers + 1
HIDDEN_SIZE = model.config.hidden_size

unique_seqs = []
seq_to_idx = {}
for r in records:
    key = tuple(r["token_ids"])
    if key not in seq_to_idx:
        seq_to_idx[key] = len(unique_seqs)
        unique_seqs.append(key)
N_UNIQUE = len(unique_seqs)
unique_lengths = np.array([len(s) for s in unique_seqs], dtype=np.int64)

pen_lookup = np.full(N_UNIQUE, -1, dtype=np.int64)
pen_idx = np.where(unique_lengths >= 3)[0]
pen_lookup[pen_idx] = np.arange(len(pen_idx))

gb = (N_STATES * (N_UNIQUE + len(pen_idx)) * HIDDEN_SIZE * 2) / 1e9
print(f"Уникальных последовательностей: {N_UNIQUE}; понадобится ~{gb:.1f} ГБ RAM под признаки")

feat_last = np.empty((N_STATES, N_UNIQUE, HIDDEN_SIZE), dtype=np.float16)
feat_pen = np.empty((N_STATES, len(pen_idx), HIDDEN_SIZE), dtype=np.float16)

order = np.argsort(unique_lengths, kind="stable")
for start in tqdm(range(0, N_UNIQUE, MODEL_BATCH_SIZE), desc="Extracting hidden states"):
    batch_ids = order[start:start + MODEL_BATCH_SIZE]
    batch_sequences = [unique_seqs[i] for i in batch_ids]
    word_lengths = np.array([len(s) for s in batch_sequences], dtype=np.int64)

    rows = [[int(BOS_ID)] + list(s) for s in batch_sequences]
    max_len = max(len(r) for r in rows)
    input_ids = torch.full((len(rows), max_len), int(PAD_ID), dtype=torch.long)
    attention_mask = torch.zeros((len(rows), max_len), dtype=torch.long)
    for i, row in enumerate(rows):
        input_ids[i, :len(row)] = torch.tensor(row, dtype=torch.long)
        attention_mask[i, :len(row)] = 1

    with torch.inference_mode():
        outputs = model(
            input_ids=input_ids.to(DEVICE),
            attention_mask=attention_mask.to(DEVICE),
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )

    row_t = torch.arange(len(rows), device=DEVICE)
    last_pos = torch.as_tensor(word_lengths, device=DEVICE)
    has_pen = word_lengths >= 3
    pen_rows = torch.as_tensor(np.where(has_pen)[0], device=DEVICE)
    pen_dest = pen_lookup[batch_ids[has_pen]]

    for layer_idx, hidden in enumerate(outputs.hidden_states):
        feat_last[layer_idx, batch_ids, :] = hidden[row_t, last_pos, :].to(torch.float16).cpu().numpy()
        if has_pen.any():
            vec = hidden[pen_rows, last_pos[pen_rows] - 1, :]
            feat_pen[layer_idx, pen_dest, :] = vec.to(torch.float16).cpu().numpy()

    del outputs
    if start % (MODEL_BATCH_SIZE * 50) == 0:
        torch.cuda.empty_cache()

n_nonfinite = int((~np.isfinite(feat_last)).sum())
print("feat_last:", feat_last.shape, "| feat_pen:", feat_pen.shape)
print("Нечисловых значений (inf/nan после float16):", n_nonfinite)

## 7. Оценить зонды по слоям

In [ ]:
real_by_len = defaultdict(list)
for r in real_records:
    real_by_len[r["num_tokens"]].append(seq_to_idx[tuple(r["token_ids"])])


def build_condition(set_name, seed):
    rng = np.random.default_rng(seed)
    pseudo_by_len = defaultdict(list)
    for r in pseudo_records:
        if r["set_name"] == set_name:
            pseudo_by_len[r["num_tokens"]].append(seq_to_idx[tuple(r["token_ids"])])
    idx, y, lens = [], [], []
    for L in sorted(real_by_len):
        p = pseudo_by_len.get(L, [])
        w = real_by_len[L]
        n = min(len(p), len(w))
        if n == 0:
            continue
        p_sel = rng.choice(p, size=n, replace=False)
        w_sel = rng.choice(w, size=n, replace=False)
        idx += list(w_sel) + list(p_sel)
        y += [1] * n + [0] * n
        lens += [L] * (2 * n)
    return np.array(idx), np.array(y), np.array(lens)


def split(y, lens, seed):
    strata = y * 10 + lens
    return train_test_split(np.arange(len(y)), test_size=TEST_SIZE, random_state=seed, stratify=strata)


def nearest_neighbors(X_train, X_test, metric, k=K_NEIGHBORS):
    X_train = X_train.astype(np.float32)
    X_test = X_test.astype(np.float32)
    if metric == "cosine":
        X_train = X_train / (np.linalg.norm(X_train, axis=1, keepdims=True) + 1e-8)
        X_test = X_test / (np.linalg.norm(X_test, axis=1, keepdims=True) + 1e-8)
        dist = -(X_test @ X_train.T)
    else:
        dist = (X_test ** 2).sum(1)[:, None] + (X_train ** 2).sum(1)[None, :] - 2.0 * (X_test @ X_train.T)
    return np.argpartition(dist, k - 1, axis=1)[:, :k]


def knn_accuracy_from_nn(nn, y_train, y_test, k=K_NEIGHBORS):
    pred = (y_train[nn].sum(axis=1) > k / 2).astype(np.int64)
    return float((pred == y_test).mean())


def linear_probe_accuracy(X_train, y_train, X_test, y_test, seed):
    n_comp = min(LINEAR_PCA_DIM, X_train.shape[0] - 1, X_train.shape[1])
    clf = make_pipeline(
        StandardScaler(),
        PCA(n_components=n_comp, random_state=seed),
        LogisticRegression(C=1.0, max_iter=2000),
    )
    clf.fit(X_train.astype(np.float32), y_train)
    return float(clf.score(X_test.astype(np.float32), y_test))


def wilson_ci(acc, n, z=1.96):
    centre = (acc + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(acc * (1 - acc) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half


CONDITIONS = [(name, s) for name, s, _ in PSEUDO_SETS]
result_rows = []
null_rows = []
condition_sizes = []

for set_name, seed in CONDITIONS:
    idx, y, lens = build_condition(set_name, seed)
    tr, te = split(y, lens, seed)
    sub = np.where(lens >= 3)[0]
    c_tr, c_te = split(y[sub], lens[sub], seed)
    c_tr, c_te = sub[c_tr], sub[c_te]
    pen_rows_all = pen_lookup[idx]
    condition_sizes.append({"set_name": set_name, "n_total": len(y), "n_test": len(te),
                            "n_len3plus": len(sub), "n_test_len3plus": len(c_te)})

    is_canon = set_name in CANON_SETS
    run_perm = set_name == CANON_SETS[0]
    rng_perm = np.random.default_rng(seed + 1000)

    for layer in tqdm(range(N_STATES), desc=f"Probing {set_name}"):
        X = feat_last[layer, idx, :]
        Xp = feat_pen[layer, pen_rows_all[c_tr], :], feat_pen[layer, pen_rows_all[c_te], :]

        def add(probe, token, acc, n_test):
            lo, hi = wilson_ci(acc, n_test)
            result_rows.append({"set_name": set_name, "seed": seed, "is_canonical_set": is_canon,
                                "layer": layer, "probe": probe, "token": token,
                                "accuracy": acc, "n_test": n_test, "wilson_low": lo, "wilson_high": hi})

        for metric in ["euclidean", "cosine"]:
            nn = nearest_neighbors(X[tr], X[te], metric)
            add(f"knn_{metric}", "last_all", knn_accuracy_from_nn(nn, y[tr], y[te]), len(te))
            if metric == "euclidean" and run_perm:
                null = [knn_accuracy_from_nn(nn, rng_perm.permutation(y[tr]), y[te])
                        for _ in range(N_PERMUTATIONS)]
                null_rows.append({"layer": layer, "null_mean": float(np.mean(null)),
                                  "null_p95": float(np.percentile(null, 95))})

            nn_c = nearest_neighbors(X[c_tr], X[c_te], metric)
            add(f"knn_{metric}", "last_len3plus", knn_accuracy_from_nn(nn_c, y[c_tr], y[c_te]), len(c_te))
            nn_p = nearest_neighbors(Xp[0], Xp[1], metric)
            add(f"knn_{metric}", "penult_len3plus", knn_accuracy_from_nn(nn_p, y[c_tr], y[c_te]), len(c_te))

        if RUN_LINEAR_PROBE and is_canon:
            add("linear", "last_all", linear_probe_accuracy(X[tr], y[tr], X[te], y[te], seed), len(te))

results_df = pd.DataFrame(result_rows)
null_df = pd.DataFrame(null_rows)
sizes_df = pd.DataFrame(condition_sizes)
results_df.to_csv(os.path.join(RESULT_DIR, "probe_accuracy_by_layer_all_runs.csv"), index=False)
null_df.to_csv(os.path.join(RESULT_DIR, "permutation_null_by_layer.csv"), index=False)
display(sizes_df)
print("Строк результатов:", len(results_df))

## 8. Сводка: средние по сидам, доверительные интервалы, пики

In [ ]:
canon = results_df[results_df["is_canonical_set"]]


def t_ci(values, conf=0.95):
    values = np.asarray(values, dtype=float)
    n = len(values)
    m = values.mean()
    if n < 2:
        return m, np.nan, np.nan
    half = stats.t.ppf(0.5 + conf / 2, n - 1) * values.std(ddof=1) / np.sqrt(n)
    return m, m - half, m + half


agg_rows = []
for (probe, token, layer), g in canon.groupby(["probe", "token", "layer"]):
    m, lo, hi = t_ci(g["accuracy"])
    agg_rows.append({"probe": probe, "token": token, "layer": layer, "mean": m,
                     "ci_low": lo, "ci_high": hi, "std": g["accuracy"].std(ddof=1), "n_seeds": len(g)})
agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(os.path.join(RESULT_DIR, "probe_accuracy_by_layer_mean_ci.csv"), index=False)

peak_rows = []
for (probe, token), g in agg_df.groupby(["probe", "token"]):
    best = g.loc[g["mean"].idxmax()]
    overlapping = g[g["ci_high"] >= best["ci_low"]]["layer"].tolist()
    peak_rows.append({
        "probe": probe, "token": token, "peak_layer": int(best["layer"]),
        "peak_mean": round(best["mean"], 3),
        "95% CI": f"[{best['ci_low']:.3f}, {best['ci_high']:.3f}]",
        "layer_0": round(g.loc[g["layer"] == 0, "mean"].item(), 3),
        "last_layer": round(g.loc[g["layer"] == g["layer"].max(), "mean"].item(), 3),
        "слои, неотличимые от пика": overlapping,
    })
peak_df = pd.DataFrame(peak_rows)
peak_df.to_csv(os.path.join(RESULT_DIR, "peak_summary.csv"), index=False)
print("Пики по основному условию (среднее по сидам):")
display(peak_df)

main = agg_df[(agg_df["probe"] == "knn_euclidean") & (agg_df["token"] == "last_all")].merge(null_df, on="layer")
above = main[main["ci_low"] > main["null_p95"]]["layer"].tolist()
print("Шумовой уровень (permutation, среднее по слоям):", round(null_df["null_mean"].mean(), 3),
      "| 95-й перцентиль, макс. по слоям:", round(null_df["null_p95"].max(), 3))
print("Слои, где нижняя граница CI выше 95-го перцентиля шума:", above)

pair_rows = []
for probe in ["knn_euclidean", "knn_cosine"]:
    for layer in range(N_STATES):
        a = canon[(canon.probe == probe) & (canon.token == "last_len3plus") & (canon.layer == layer)].sort_values("seed")["accuracy"].values
        b = canon[(canon.probe == probe) & (canon.token == "penult_len3plus") & (canon.layer == layer)].sort_values("seed")["accuracy"].values
        diff = a - b
        p = stats.ttest_rel(a, b).pvalue if len(a) > 1 and np.std(diff) > 0 else np.nan
        pair_rows.append({"probe": probe, "layer": layer, "last_minus_penult": diff.mean(), "p_value": p})
pair_df = pd.DataFrame(pair_rows)
pair_df.to_csv(os.path.join(RESULT_DIR, "last_vs_penultimate.csv"), index=False)
sig = pair_df[(pair_df.probe == "knn_euclidean") & (pair_df.p_value < 0.05)]
print("Евклидов k-NN: слои, где последний и предпоследний токены достоверно различаются (p<0.05, без поправки):")
display(sig.round(3))

first = results_df[(results_df.probe == "knn_euclidean") & (results_df.token == "last_all")]
cmp = (first[first.set_name.isin([CANON_SETS[0], NOFILTER_SET])]
       .pivot(index="layer", columns="set_name", values="accuracy"))
cmp["разница (без фильтра − канон.)"] = cmp[NOFILTER_SET] - cmp[CANON_SETS[0]]
print(f"Евклидов k-NN, последний токен: {NOFILTER_SET} против {CANON_SETS[0]}")
display(cmp.round(3).T)

## 9. Графики

In [ ]:
COLORS = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "violet": "#4a3aa7"}
TEXT2 = "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": TEXT2, "axes.labelcolor": "#0b0b0b",
                     "xtick.color": TEXT2, "ytick.color": TEXT2, "font.size": 10})


def curve(probe, token):
    return agg_df[(agg_df.probe == probe) & (agg_df.token == token)].sort_values("layer")


def draw(ax, df, color, label, marker="o"):
    ax.plot(df["layer"], df["mean"], color=color, lw=2, marker=marker, ms=3, label=label)
    ax.fill_between(df["layer"], df["ci_low"], df["ci_high"], color=color, alpha=0.18, lw=0)


def finish(ax, title):
    ax.axhline(0.5, ls="--", lw=1, color=TEXT2)
    ax.set_xlabel("Слой (0 = embedding)")
    ax.set_ylabel("Точность")
    ax.set_ylim(0.35, 1.0)
    ax.grid(alpha=0.25)
    ax.set_title(title, loc="left", fontsize=11)
    ax.legend(frameon=False, fontsize=9, loc="lower right")


fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
n_s = len(CANON_SETS)

ax = axes[0, 0]
ax.fill_between(null_df["layer"], 0, null_df["null_p95"], color="#bdbcb7", alpha=0.45, lw=0,
                label="Шум: 95-й перцентиль (перемешанные метки)")
draw(ax, curve("knn_euclidean", "last_all"), COLORS["blue"], f"Последний токен, среднее ± 95% CI ({n_s} сидов)")
finish(ax, "1. Слова и псевдослова различимы (евклидов k-NN)")

ax = axes[0, 1]
draw(ax, curve("knn_euclidean", "last_len3plus"), COLORS["blue"], "Последний токен")
draw(ax, curve("knn_euclidean", "penult_len3plus"), COLORS["orange"], "Предпоследний токен", marker="s")
finish(ax, "2. Контроль: слова из 3–4 токенов")

ax = axes[1, 0]
one = results_df[(results_df.probe == "knn_euclidean") & (results_df.token == "last_all")]
for name, color, label, mk in [(NOFILTER_SET, COLORS["orange"], "Псевдослова без фильтров (версия 1)", "s"),
                               (CANON_SETS[0], COLORS["blue"], "Канонические псевдослова", "o")]:
    d = one[one.set_name == name].sort_values("layer")
    ax.plot(d["layer"], d["accuracy"], color=color, lw=2, marker=mk, ms=3, label=label)
    ax.fill_between(d["layer"], d["wilson_low"], d["wilson_high"], color=color, alpha=0.15, lw=0)
finish(ax, f"3. Влияние фильтров (сид {SEEDS[0]}, интервал Уилсона)")

ax = axes[1, 1]
draw(ax, curve("knn_euclidean", "last_all"), COLORS["blue"], "k-NN, евклидово расстояние")
draw(ax, curve("knn_cosine", "last_all"), COLORS["aqua"], "k-NN, косинусное расстояние", marker="s")
if RUN_LINEAR_PROBE:
    draw(ax, curve("linear", "last_all"), COLORS["violet"], "Линейный зонд (PCA + логрегрессия)", marker="^")
finish(ax, "4. Сравнение зондов (последний токен)")

fig.suptitle("Saiga Llama 3 8B: различение русских слов и псевдослов по слоям", x=0.01, ha="left", fontsize=13)
fig.tight_layout()
plot_path = os.path.join(RESULT_DIR, "probe_accuracy_by_layer.png")
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
plt.show()
print("График:", plot_path)

## 10. Контроль: та же модель, английские слова

### Английские слова и псевдослова

In [ ]:
for _name in ["tokenizer", "model", "is_canonical", "selected_words", "nearest_neighbors",
              "knn_accuracy_from_nn", "split", "agg_df"]:
    assert _name in globals(), f"Не найдено {_name}: сначала выполните весь основной ноутбук"

EN_SEEDS = SEEDS[:2]
rng_en = np.random.default_rng(SEED)

en_forms, en_words = set(), []
for w in top_n_list("en", TOP_N_WORDS):
    w = w.strip().lower()
    if re.fullmatch(r"[a-z]+", w) and len(w) >= 3 and w not in en_forms:
        en_forms.add(w)
        en_words.append(w)

en_by_len, en_known = defaultdict(list), set()
for start in range(0, len(en_words), TOKENIZE_BATCH_SIZE):
    batch = en_words[start:start + TOKENIZE_BATCH_SIZE]
    for w, ids in zip(batch, tokenizer([" " + x for x in batch], add_special_tokens=False)["input_ids"]):
        if MIN_TOKENS <= len(ids) <= MAX_TOKENS and tuple(ids) not in en_known:
            en_known.add(tuple(ids))
            en_by_len[len(ids)].append({"word": w, "token_ids": list(ids)})

ru_len_counts = Counter(x["num_tokens"] for x in selected_words)
en_selected = {}
for L, n in sorted(ru_len_counts.items()):
    pool = en_by_len[L]
    en_selected[L] = [pool[i] for i in rng_en.permutation(len(pool))[:n]]
    if len(en_selected[L]) < n:
        print(f"Внимание: английских слов из {L} токенов только {len(en_selected[L])} (нужно {n})")


def make_en_pseudowords(seed):
    rng = np.random.default_rng(seed)
    out, seen = {}, set()
    for L, group in en_selected.items():
        pools = [np.array([g["token_ids"][p] for g in group]) for p in range(L)]
        out[L] = []
        for _ in range(len(group)):
            for _attempt in range(MAX_ATTEMPTS):
                cand = tuple(int(rng.choice(pool)) for pool in pools)
                if cand in en_known or cand in seen:
                    continue
                text = tokenizer.decode(list(cand), clean_up_tokenization_spaces=False).strip()
                if not re.fullmatch(r"[a-z]+", text) or not is_canonical(cand) or text in en_forms:
                    continue
                seen.add(cand)
                out[L].append(list(cand))
                break
    return out


en_pseudo = {s: make_en_pseudowords(s) for s in EN_SEEDS}
print("Английских слов по длинам:", {L: len(v) for L, v in en_selected.items()})
for s in EN_SEEDS:
    print(f"Псевдослов, сид {s}:", {L: len(v) for L, v in en_pseudo[s].items()})
print("Примеры слов:", ", ".join(w["word"] for L in en_selected for w in en_selected[L][:4]))
print("Примеры псевдослов:", ", ".join(
    tokenizer.decode(p).strip() for L in en_pseudo[EN_SEEDS[0]] for p in en_pseudo[EN_SEEDS[0]][L][:4]))

### Векторы последнего и предпоследнего токенов

In [ ]:
en_seqs, en_seq_idx = [], {}
for seq in ([w["token_ids"] for L in en_selected for w in en_selected[L]]
            + [p for s in EN_SEEDS for L in en_pseudo[s] for p in en_pseudo[s][L]]):
    if tuple(seq) not in en_seq_idx:
        en_seq_idx[tuple(seq)] = len(en_seqs)
        en_seqs.append(list(seq))
en_lens = np.array([len(s) for s in en_seqs])

en_last = np.empty((N_STATES, len(en_seqs), HIDDEN_SIZE), dtype=np.float16)
en_pen = np.zeros((N_STATES, len(en_seqs), HIDDEN_SIZE), dtype=np.float16)
for start in tqdm(range(0, len(en_seqs), MODEL_BATCH_SIZE), desc="English hidden states"):
    ids_b = np.argsort(en_lens, kind="stable")[start:start + MODEL_BATCH_SIZE]
    rows = [[int(BOS_ID)] + en_seqs[i] for i in ids_b]
    max_len = max(len(r) for r in rows)
    input_ids = torch.full((len(rows), max_len), int(PAD_ID), dtype=torch.long)
    mask = torch.zeros((len(rows), max_len), dtype=torch.long)
    for i, r in enumerate(rows):
        input_ids[i, :len(r)] = torch.tensor(r)
        mask[i, :len(r)] = 1
    with torch.inference_mode():
        out = model(input_ids=input_ids.to(DEVICE), attention_mask=mask.to(DEVICE),
                    output_hidden_states=True, use_cache=False, return_dict=True)
    r_t = torch.arange(len(rows), device=DEVICE)
    last_pos = torch.as_tensor(en_lens[ids_b], device=DEVICE)
    for layer, h in enumerate(out.hidden_states):
        en_last[layer, ids_b] = h[r_t, last_pos].to(torch.float16).cpu().numpy()
        en_pen[layer, ids_b] = h[r_t, last_pos - 1].to(torch.float16).cpu().numpy()
    del out
print("Готово:", en_last.shape)

### Оценка и сравнение с русским

In [ ]:
en_rows = []
for s in EN_SEEDS:
    rng_c = np.random.default_rng(s)
    idx, y, lens = [], [], []
    for L in sorted(en_selected):
        w = [en_seq_idx[tuple(x["token_ids"])] for x in en_selected[L]]
        p = [en_seq_idx[tuple(x)] for x in en_pseudo[s][L]]
        n = min(len(w), len(p))
        idx += list(rng_c.choice(w, n, replace=False)) + list(rng_c.choice(p, n, replace=False))
        y += [1] * n + [0] * n
        lens += [L] * (2 * n)
    idx, y, lens = np.array(idx), np.array(y), np.array(lens)
    tr, te = split(y, lens, s)
    sub = np.where(lens >= 3)[0]
    c_tr, c_te = split(y[sub], lens[sub], s)
    c_tr, c_te = sub[c_tr], sub[c_te]
    for layer in range(N_STATES):
        X, P = en_last[layer, idx], en_pen[layer, idx]
        for token, Xa, a_tr, a_te in [("last_all", X, tr, te), ("last_len3plus", X, c_tr, c_te),
                                      ("penult_len3plus", P, c_tr, c_te)]:
            nn = nearest_neighbors(Xa[a_tr], Xa[a_te], "euclidean")
            en_rows.append({"seed": s, "layer": layer, "token": token,
                            "accuracy": knn_accuracy_from_nn(nn, y[a_tr], y[a_te])})

en_df = pd.DataFrame(en_rows)
en_mean = en_df.groupby(["token", "layer"])["accuracy"].mean().reset_index()
ru_mean = agg_df[agg_df.probe == "knn_euclidean"][["token", "layer", "mean"]].rename(columns={"mean": "accuracy"})
en_df.to_csv(os.path.join(RESULT_DIR, "english_check_all_runs.csv"), index=False)


def describe(df, lang):
    get = lambda t: df[df.token == t].set_index("layer")["accuracy"]
    last_all, last3, pen3 = get("last_all"), get("last_len3plus"), get("penult_len3plus")
    late = slice(16, N_STATES - 1)
    return {
        "Язык": lang,
        "Слой 0": round(last_all.loc[0], 3),
        "Последний токен (все): пик": f"{last_all.max():.3f} (слой {last_all.idxmax()})",
        "Последний (3–4 токена): пик": f"{last3.max():.3f} (слой {last3.idxmax()})",
        "Предпоследний (3–4 токена): пик": f"{pen3.max():.3f} (слой {pen3.idxmax()})",
        "Пик: последний − предпоследний": round(last3.max() - pen3.max(), 3),
        "Слои 16–32: последний − предпоследний": round((last3.loc[late] - pen3.loc[late]).mean(), 3),
    }


cmp_df = pd.DataFrame([describe(ru_mean, f"русский ({len(SEEDS)} сидов)"),
                       describe(en_mean, f"английский ({len(EN_SEEDS)} сида)")])
display(cmp_df)
cmp_df.to_csv(os.path.join(RESULT_DIR, "english_vs_russian.csv"), index=False)
print("Для справки, статья (Llama2-7B, английский): последний токен 89%, предпоследний 61%.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (df, title) in zip(axes, [(ru_mean, "Русский"), (en_mean, "Английский")]):
    for token, color, label in [("last_all", "#2a78d6", "Последний токен, все слова"),
                                ("last_len3plus", "#1baf7a", "Последний токен, 3–4 токена"),
                                ("penult_len3plus", "#eb6834", "Предпоследний токен, 3–4 токена")]:
        d = df[df.token == token].sort_values("layer")
        ax.plot(d["layer"], d["accuracy"], color=color, lw=2, marker="o", ms=3, label=label)
    ax.axhline(0.5, ls="--", lw=1, color="#52514e")
    ax.set_title(f"{title}: Saiga, евклидов k-NN", loc="left", fontsize=11)
    ax.set_xlabel("Слой (0 = embedding)")
    ax.set_ylim(0.35, 1.0)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=9, loc="lower right")
axes[0].set_ylabel("Точность")
fig.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, "english_vs_russian.png"), dpi=180, bbox_inches="tight")
plt.show()

## 11. Скачать результаты

In [ ]:
from google.colab import files
for name in [
    "russian_words_and_pseudowords.csv",
    "pseudoword_generation_stats.csv",
    "probe_accuracy_by_layer_all_runs.csv",
    "probe_accuracy_by_layer_mean_ci.csv",
    "peak_summary.csv",
    "last_vs_penultimate.csv",
    "permutation_null_by_layer.csv",
    "probe_accuracy_by_layer.png",
]:
    files.download(os.path.join(RESULT_DIR, name))